# Economic Load Dispatch with Prohibited Operating Zones

This notebook runs on a local simulator throughout.

## Overview

A dispatch has a fixed demand, a fuel cost and a prohibited operating zone per generator. The notebook takes the fuel cost as the objective of a `BinaryOptimizationProblem`, states the demand and the zones as linear constraints, and runs PCE-VQE on it, which encodes the binary variables with `PolyEncoding`. A six-generator variant at the end uses the same workflow.

## Local example (3 generators, 12 decision variables)

In [ ]:
import time

from economic_load_dispatch import (
    build_problem,
    define_generators,
    define_generators_large,
    find_best_repaired_solution,
)
from qiskit.circuit.library import CXGate, RYGate, RZGate

from divi.backends import MaestroSimulator
from divi.qprog.algorithms import PCE, GenericLayerAnsatz, PolyEncoding, SoftObjective
from divi.qprog.optimizers import PymooMethod, PymooOptimizer

### Step 1 — Define the generators

Each generator has a quadratic fuel cost curve, operating limits and a
prohibited operating zone.

In [ ]:
DEMAND = 195  # MW

generators = define_generators()

print(f"3 generators, demand = {DEMAND} MW")
for g in generators:
    print(
        f"  {g['name']}: {g['P_min']}–{g['P_max']} MW, POZ [{g['poz_low']},{g['poz_high']}]"
    )

### Step 2 — Build the problem

The objective is the fuel cost. Total generation must equal the demand (an equality constraint), and each generator's second-highest bit may not be set while its highest bit is clear (an inequality constraint, penalised, with one slack variable each). A violation of `v` costs `v**2` times the penalty weight.

In [ ]:
problem = build_problem(generators, demand=DEMAND)
n_variables = len(problem.canonical_problem.variable_order)
print(f"Built problem: {n_variables} variables, slack included")

### Step 3 — Solve with PCE-VQE (locally)

In [ ]:
backend = MaestroSimulator(shots=10_000)
ansatz = GenericLayerAnsatz(
    gate_sequence=[RYGate, RZGate], entangler=CXGate, entangling_layout="all-to-all"
)

t0 = time.time()
pce_solver = PCE(
    problem,
    ansatz=ansatz,
    n_layers=3,
    encoding=PolyEncoding(),
    optimizer=PymooOptimizer(method=PymooMethod.DE, population_size=30),
    max_iterations=20,
    objective=SoftObjective(alpha=3.0),
    backend=backend,
)
print(f"PCE qubits: {pce_solver.n_qubits} (poly encoding of {n_variables} variables)")
pce_solver.run()
local_time = time.time() - t0
print(f"Solved in {local_time:.1f}s")

### Step 4 — Decode and repair

In [ ]:
best = find_best_repaired_solution(pce_solver, generators, DEMAND)

if best:
    powers, cost, prob = best
    gen_str = ", ".join(f"P{i + 1}={p:.0f}" for i, p in enumerate(powers))
    print(f"Best repaired solution: {gen_str} MW")
    print(
        f"Total={sum(powers):.0f} MW, Cost={cost:.1f} $, quantum seed prob={prob:.2%}"
    )

---

## Six-generator variant

Six generators.

In [ ]:
generators_large = define_generators_large()
DEMAND_LARGE = 390

problem_large = build_problem(generators_large, demand=DEMAND_LARGE)
n_variables_large = len(problem_large.canonical_problem.variable_order)
print(f"Built problem: {n_variables_large} variables (was {n_variables})")

In [ ]:
ansatz_large = GenericLayerAnsatz(
    gate_sequence=[RYGate, RZGate], entangler=CXGate, entangling_layout="all-to-all"
)

t0 = time.time()
pce_large = PCE(
    problem_large,
    ansatz=ansatz_large,
    n_layers=3,
    encoding=PolyEncoding(),
    optimizer=PymooOptimizer(method=PymooMethod.DE, population_size=30),
    max_iterations=10,
    objective=SoftObjective(alpha=3.0),
    backend=backend,
)
print(
    f"PCE qubits: {pce_large.n_qubits} (poly encoding of {n_variables_large} variables)"
)
pce_large.run()
large_time = time.time() - t0
print(f"Six-generator solve completed in {large_time:.1f}s")

In [ ]:
best_large = find_best_repaired_solution(pce_large, generators_large, DEMAND_LARGE)

if best_large:
    powers, cost, prob = best_large
    gen_str = ", ".join(f"P{i + 1}={p:.0f}" for i, p in enumerate(powers))
    print(f"Best repaired solution: {gen_str} MW")
    print(
        f"Total={sum(powers):.0f} MW, Cost={cost:.1f} $, quantum seed prob={prob:.2%}"
    )

print(
    f"Six-generator run: {large_time:.1f}s for 6 generators ({n_variables_large} vars)"
)

---

## Explore further

The notebook runs the workflow for three and for six generators.

Change the demand or generator parameters to explore the formulation.